# Compute Routing

## What you'll learn

- Understand the difference between step runners and compute routing
- Switch a step's compute provider with the `compute_provider` parameter
- Use reusable operation defaults for compute routing
- Mix compute providers in a single pipeline
- Know which operations can run remotely (command ops)

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Step Overrides](../05-errors-and-control/01-step-overrides.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

In [ ]:
env = tutorial_setup("compute_routing")

## Step runner and compute provider

The **step runner** schedules the worker that prepares inputs and records results.
The **compute provider** chooses where that worker sends the execute phase.
A local worker can call a remote Modal endpoint, then collect the result locally.

This tutorial uses local compute throughout. The optional
[Running on Modal](04-modal-execution.ipynb) tutorial deploys a command operation
and sends its execute phase to Modal.

## The `compute_provider` parameter

Every `pipeline.run()` and `pipeline.submit()` call accepts a `compute_provider`
parameter. Set it to route the execute phase to a different provider. Everything
else — operation class, inputs, params, output wiring — stays identical.

```python
# Use the operation class default
step = pipeline.run(MyOp, inputs=...)

# Modal — same operation, same inputs, different compute provider
step = pipeline.run(MyOp, inputs=..., compute_provider="modal")
```

For a command operation with a deployed endpoint, the step can select Modal.
The plain function operations below use local compute. The command-wrapper
option for Python functions is covered later.

In [ ]:
pipeline = PipelineManager.create(
    name="compute_routing_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

print(f"DataGenerator default provider: {DataGenerator().compute_provider.active}")

### Generate data (local compute)

DataGenerator is fast, so its operation class keeps the local compute default.

In [ ]:
step0 = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
)
generated = inspect_step(
    env.delta_root, step0.step_number, pipeline_run_id=pipeline.config.pipeline_run_id
)
assert generated.height == 4
print(f"Generated {generated.height} datasets with the operation default")

### Transform data

Select `"local"` explicitly for this step. The override applies to this invocation;
without it, the operation’s declared compute provider is used.

With two variants per input, four input datasets should produce eight outputs.

In [ ]:
step1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 2, "seed": 100},
    compute_provider="local",
)
transformed = inspect_step(
    env.delta_root, step1.step_number, pipeline_run_id=pipeline.config.pipeline_run_id
)
assert step1.succeeded_count == 4
assert transformed.height == 8
print(f"Processed {step1.succeeded_count} inputs into {transformed.height} datasets")

In [ ]:
step2 = pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    compute_provider="local",
)
print(f"Computed metrics for {step2.succeeded_count} datasets with local compute")

In [ ]:
summary = pipeline.finalize()
print(
    f"Pipeline complete: {summary['total_steps']} steps, "
    f"success={summary['overall_success']}"
)
inspect_pipeline(env.delta_root)

## Operation defaults and explicit step overrides

Each operation class declares its reusable compute configuration. When a step
omits `compute_provider`, that declaration remains in effect. An explicit step
value selects or patches the provider for that invocation.

Compute-routing precedence has two levels:

```
Operation class default  →  Explicit run/submit override (wins)
```

In [ ]:
pipeline2 = PipelineManager.create(
    name="compute_defaults_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output2 = pipeline2.output

# These steps use their operation-class compute configuration.
pipeline2.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
)
pipeline2.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output2("generate", "datasets")},
)

# This step supplies an explicit provider for this invocation.
pipeline2.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output2("transform", "dataset")},
    compute_provider="local",  # would be "modal" for a deployed command op
)

pipeline2.finalize()
print(f"DataGenerator operation default: {DataGenerator().compute_provider.active}")
print("First two used operation defaults; the last supplied a step override")

## What can run on Modal: command operations

The Modal provider runs **command operations**: either an explicit
`ToolSpec` plus `execute_command()`, or a Python function operation that
opts into a command wrapper with `execute_as_tool=True`. The
framework's endpoint router ships the op's `Params` + input files to the
tool's deployed endpoint, which runs the command in a container and
returns the output files.

A plain `execute_function()` operation without that opt-in runs locally.
Routing it to `"modal"` fails configuration validation.

See [Running on Modal](04-modal-execution.ipynb) for the tool-op anatomy
and deployment.

In [ ]:
from artisan.operations.examples import WaitTool

print(
    f"WaitTool (ToolSpec + execute_command): is_command_op = {WaitTool().is_command_op()}"
)
print(
    f"DataGenerator (custom execute):      is_command_op = {DataGenerator().is_command_op()}"
)
print("\nOnly command ops can route to 'modal'; pure-Python ops run locally.")

## Summary

| Concept | What it does |
|---------|-------------|
| Step runner (`step_runner`) | Controls where the *worker process* runs (local or through an optional provider) |
| Operation compute default | Reusable base routing declared by the operation class |
| Step `compute_provider` | Explicitly selects or patches routing for one invocation |
| Command operations | Explicit `ToolSpec` + `execute_command()`, or the `execute_as_tool=True` wrapper, can use Modal |

The runner and compute provider have separate responsibilities.
Declare reusable routing on the operation, then pass `compute_provider="modal"`
only for steps that should use a deployed remote endpoint.

## Next steps

- [Running on Modal](04-modal-execution.ipynb) — Tool endpoints: deploy, run, and debug
- [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb) — Apply configuration to individual steps
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Recipes for execution configuration